# 06 - Limpieza y preparación ML

* Se conservan los pares producto-destino con alguna exportación positiva
* Alguna brecha en USD positiva entre 2017 y 2024.
* Se excluyen los productos minero-energéticos.

## Importar librerías

In [1]:
from pathlib import Path
from zipfile import ZipFile
from IPython.display import display
import json
import numpy as np
import pandas as pd
import duckdb

## Mejorar la visualización

In [2]:
# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

## Establecer las rutas

In [3]:
RAIZ = Path.cwd().resolve()
for carpeta in [RAIZ, *RAIZ.parents]:
    if (carpeta / 'requirements.txt').is_file() and (carpeta / 'data').is_dir():
        RAIZ = carpeta
        break
else:
    raise FileNotFoundError('Abra el notebook desde el proyecto.')

RAW = RAIZ / 'data/raw'
PROCESSED = RAIZ / 'data/processed'
ANIOS = list(range(2017, 2025))
archivo_panel = PROCESSED / 'gravitacional/panel_gravitacional_COL_2017_2024_HS2017.parquet'
catalogo_destinos = pd.read_parquet(PROCESSED / 'variables/catalogo_destinos_variables.parquet')
catalogo_productos = pd.read_csv(RAW / 'baci/product_codes_HS17_V202601.csv', dtype = {'code': str})
registro_importacion = json.loads((RAIZ / 'data/reports/05_importacion_ml/importacion.json').read_text(encoding='utf-8'))

## Filtrar los pares producto-destino

In [4]:
with duckdb.connect() as con:
    con.execute("SET threads=2")
    con.execute("SET memory_limit='512MB'")
    base_filtrada = con.execute("""WITH pares AS (
        SELECT hs6, iso3_d FROM read_parquet(?)
        GROUP BY hs6, iso3_d
        HAVING bool_or(trade_value_usd>0) AND bool_or(brecha_usd>0)
    ) SELECT p.iso3_o, p.hs6, p.iso3_d, p.year, p.trade_value_usd,
        p.registro_baci, p.cero_construido, p.exportaciones_estimadas_usd,
        p.brecha_usd, p.brecha_relativa, p.efecto_fijo_finito, p.estado_ajuste
        FROM read_parquet(?) p JOIN pares USING(hs6, iso3_d)
        WHERE left(p.hs6,2) NOT IN ('25','26','27')
        AND left(p.hs6,4) NOT IN ('7102','7103','7106','7108','7110')
        AND p.hs6<>'720260' ORDER BY p.hs6,p.iso3_d,p.year""", [str(archivo_panel), str(archivo_panel)]).df()
base_filtrada = base_filtrada.merge(catalogo_productos[['code', 'description']].rename(columns={
    'code':'hs6', 'description':'nombre_producto_hs6_baci'}), on='hs6', how='left', validate='many_to_one')
base_filtrada = base_filtrada.merge(catalogo_destinos[['iso3_d', 'nombre_destino']].rename(columns={
    'nombre_destino':'nombre_pais_destino'}), on='iso3_d', how='left', validate='many_to_one')
assert base_filtrada.groupby(['hs6', 'iso3_d'])['year'].nunique().eq(8).all()

In [5]:
base_filtrada.head()

,iso3_o,hs6,iso3_d,year,trade_value_usd,registro_baci,cero_construido,exportaciones_estimadas_usd,brecha_usd,brecha_relativa,efecto_fijo_finito,estado_ajuste,nombre_producto_hs6_baci,nombre_pais_destino
0,COL,010121,ABW,2017,0.00,False,True,421.96,421.96,1.00,True,ajuste_en_muestra,"Horses: live, pure-bred breeding animals",Aruba
1,COL,010121,ABW,2018,0.00,False,True,"1,214.99","1,214.99",1.00,True,ajuste_en_muestra,"Horses: live, pure-bred breeding animals",Aruba
2,COL,010121,ABW,2019,"4,300.00",True,False,435.60,"-3,864.40",-8.87,True,ajuste_en_muestra,"Horses: live, pure-bred breeding animals",Aruba
3,COL,010121,ABW,2020,0.00,False,True,382.85,382.85,1.00,True,ajuste_en_muestra,"Horses: live, pure-bred breeding animals",Aruba
4,COL,010121,ABW,2021,0.00,False,True,591.10,591.10,1.00,True,ajuste_en_muestra,"Horses: live, pure-bred breeding animals",Aruba


In [6]:
base_filtrada.shape

(875320, 14)

## Preparar la TRM anual

In [7]:
carpeta_precios = RAW / 'variables_ml/precios_trm'
trm = pd.DataFrame(json.loads((carpeta_precios / 'trm_sfc_2017_2024.json').read_text(encoding='utf-8')))
trm['valor'] = pd.to_numeric(trm['valor'], errors = 'raise')
assert trm['unidad'].eq('COP').all() and trm['valor'].gt(0).all()
for columna in ['vigenciadesde', 'vigenciahasta']:
    trm[columna] = pd.to_datetime(trm[columna], errors = 'raise')
assert trm['vigenciahasta'].ge(trm['vigenciadesde']).all()
dias = []
for fila in trm.itertuples(index = False):
    fechas = pd.date_range(max(fila.vigenciadesde, pd.Timestamp('2017-01-01')),
        min(fila.vigenciahasta, pd.Timestamp('2024-12-31')))
    dias.append(pd.DataFrame({'fecha': fechas, 'trm_cop_por_usd': float(fila.valor)}))
trm_diaria = pd.concat(dias, ignore_index = True).sort_values('fecha')
assert not trm_diaria['fecha'].duplicated().any()
assert pd.DatetimeIndex(trm_diaria['fecha']).equals(pd.date_range('2017-01-01', '2024-12-31'))
trm_diaria['trm_usd_por_cop'] = 1 / trm_diaria['trm_cop_por_usd']
trm_diaria['year'] = trm_diaria['fecha'].dt.year
trm_anual = trm_diaria.groupby('year', as_index = False)[['trm_cop_por_usd', 'trm_usd_por_cop']].mean()

## Convertir el contexto productivo de COP a USD

In [8]:
contexto_dane = pd.read_parquet(RAW / 'variables_ml/complementos/contexto_dane_hs6_anio.parquet')
contexto_dane = contexto_dane.merge(trm_anual[['year', 'trm_usd_por_cop']], on = 'year', validate = 'many_to_one')
# La unidad original DANE es miles de millones de COP; la salida es USD corrientes.
contexto_dane['va_sector_col_usd'] = contexto_dane['va_sector_col_miles_millones_cop'] * 1_000_000_000 * contexto_dane['trm_usd_por_cop']
contexto_dane['ci_sector_col_usd'] = contexto_dane['ci_sector_col_miles_millones_cop'] * 1_000_000_000 * contexto_dane['trm_usd_por_cop']
contexto_dane[['hs6', 'year', 'va_sector_col_miles_millones_cop', 'ci_sector_col_miles_millones_cop',
    'trm_usd_por_cop', 'va_sector_col_usd', 'ci_sector_col_usd']].head()

,hs6,year,va_sector_col_miles_millones_cop,ci_sector_col_miles_millones_cop,trm_usd_por_cop,va_sector_col_usd,ci_sector_col_usd
0,010121,2017,"14,639.00","16,149.00",0.00,"4,961,836,144.02","5,473,645,186.81"
1,010121,2018,"15,224.00","16,788.00",0.00,"5,161,350,126.73","5,691,588,671.01"
2,010121,2019,"15,735.00","17,787.00",0.00,"4,802,612,775.73","5,428,921,095.76"
3,010121,2020,"16,980.00","18,498.00",0.00,"4,613,056,801.56","5,025,460,819.51"
4,010121,2021,"21,931.00","22,560.00",0.00,"5,868,190,837.58","6,036,495,613.32"


## Preparar los precios internacionales

In [9]:
precios = pd.read_excel(carpeta_precios / 'CMO-Historical-Data-Annual.xlsx',
    sheet_name = 'Annual Prices (Nominal)', header = 6,
    usecols = ['Unnamed: 0', 'Coffee, Arabica', 'Crude oil, Brent'])
assert precios.iloc[0]['Coffee, Arabica'] == '($/kg)' and precios.iloc[0]['Crude oil, Brent'] == '($/bbl)'
precios = precios.loc[pd.to_numeric(precios['Unnamed: 0'], errors='coerce').isin(ANIOS)].rename(columns={
    'Unnamed: 0':'year', 'Coffee, Arabica':'precio_cafe_arabica_usd_kg',
    'Crude oil, Brent':'precio_petroleo_brent_usd_barril'})
precios['year'] = precios['year'].astype(int)
for columna in ['precio_cafe_arabica_usd_kg', 'precio_petroleo_brent_usd_barril']:
    precios[columna] = pd.to_numeric(precios[columna], errors = 'raise')
series_anuales = trm_anual.merge(precios, on = 'year', validate = 'one_to_one')
assert set(series_anuales['year']) == set(ANIOS)

## Preparar las variables de destino

In [10]:
indicadores = [
    ('NY.GDP.MKTP.PP.KD', 'pib_ppa_constante_2021'), ('SP.POP.TOTL', 'poblacion_total'),
    ('NV.IND.TOTL.KD', 'va_industrial_constante_2015_usd'), ('BX.KLT.DINV.WD.GD.ZS', 'ied_entrada_pct_pib'),
    ('EG.ELC.ACCS.ZS', 'acceso_electricidad_pct'), ('GOV_WGI_PV.EST', 'wgi_estabilidad_politica')
]
fuentes_wb = [ruta for ruta, detalle in registro_importacion['archivos'].items()
    if detalle['fuente'] in ['Banco Mundial - WDI', 'Banco Mundial - WGI']]
registros_wb = []
for ruta in fuentes_wb:
    registros_wb.extend(json.loads((RAIZ / ruta).read_text(encoding='utf-8'))['registros'])
destinos_ml = catalogo_destinos.loc[catalogo_destinos['iso3_d'].isin(base_filtrada['iso3_d']), ['iso3_d', 'iso3_referencia']]
datos_destino = destinos_ml.merge(pd.DataFrame({'year': ANIOS}), how = 'cross')
wgi_sin_iso = {'Taiwan, China':'TWN', 'Cook Islands':'COK', 'Niue':'NIU', 'French Guiana':'GUF',
    'Martinique':'MTQ', 'Reunion':'REU', 'Jersey':'JEY', 'Anguilla':'AIA', 'Guadeloupe':'GLP'}
for codigo, variable in indicadores:
    filas = []
    for registro in registros_wb:
        if registro['indicator']['id'] != codigo:
            continue
        iso = registro['countryiso3code'] or wgi_sin_iso.get(registro['country']['value'])
        if iso in set(destinos_ml['iso3_referencia']) and int(registro['date']) in ANIOS:
            filas.append({'iso3_referencia': iso, 'year': int(registro['date']), variable: registro['value']})
    serie = pd.DataFrame(filas)
    serie[variable] = pd.to_numeric(serie[variable], errors = 'raise')
    datos_destino = datos_destino.merge(serie, on = ['iso3_referencia', 'year'], how = 'left', validate = 'many_to_one')
datos_destino.drop(columns = 'iso3_referencia', inplace = True)

## Preparar las importaciones y proveedores por producto

In [11]:
pares_mercado = base_filtrada[['hs6', 'iso3_d']].drop_duplicates().merge(
    catalogo_destinos[['iso3_d', 'codigo_baci']], on = 'iso3_d', validate = 'many_to_one')
partes_mercado = []
with duckdb.connect() as con:
    con.execute("SET threads=2")
    con.execute("SET memory_limit='512MB'")
    con.register('pares_mercado', pares_mercado)
    for anio in ANIOS:
        con.execute('CREATE OR REPLACE TEMP TABLE comercio_crudo(k VARCHAR, i INTEGER, j INTEGER, v DOUBLE)')
        with ZipFile(RAW / 'baci/BACI_HS17_V202601.zip') as paquete:
            with paquete.open(f'BACI_HS17_Y{anio}_V202601.csv') as entrada:
                for bloque in pd.read_csv(entrada, usecols=['k','i','j','v'], dtype={'k':'string'}, float_precision='round_trip', chunksize=500_000):
                    bloque['k'] = bloque['k'].str.strip().str.zfill(6)
                    seleccion = bloque.merge(pares_mercado[['hs6','codigo_baci']], left_on=['k','j'],
                        right_on=['hs6','codigo_baci'], how='inner', validate='many_to_one')
                    con.register('bloque_baci', seleccion)
                    con.execute('INSERT INTO comercio_crudo SELECT k,i,j,v FROM bloque_baci')
                    con.unregister('bloque_baci')
        consulta = """SELECT p.hs6,p.iso3_d,
            sum(b.v*1000) AS importaciones_producto_total_usd,
            count(DISTINCT b.i) FILTER(WHERE b.i<>170) AS proveedores_ex_col
            FROM comercio_crudo b JOIN pares_mercado p ON b.k=p.hs6 AND b.j=p.codigo_baci
            WHERE b.v>0 GROUP BY p.hs6,p.iso3_d"""
        partes_mercado.append(con.execute(consulta).df().assign(year = anio))
mercado = pd.concat(partes_mercado, ignore_index = True)

## Preparar la participación de la oferta colombiana

In [12]:
baci_colombia = pd.read_parquet(PROCESSED / 'baci/baci_exports_COL_2017_2024_HS2017_V202601.parquet',
    columns = ['hs6', 'year', 'trade_value_usd'])
total_colombia = baci_colombia.groupby('year')['trade_value_usd'].sum()
exportaciones_producto = baci_colombia.groupby(['hs6', 'year'])['trade_value_usd'].sum()
grilla = pd.MultiIndex.from_product([sorted(baci_colombia['hs6'].unique()), ANIOS], names = ['hs6', 'year'])
oferta = exportaciones_producto.reindex(grilla, fill_value = 0).rename('oferta_exportada_col_usd').reset_index()
oferta['participacion_oferta_col'] = oferta['oferta_exportada_col_usd'] / oferta['year'].map(total_colombia)

## Preparar el calendario presidencial

In [13]:
calendario = pd.DataFrame({'year': ANIOS})
anios_electorales = [fila['year'] for fila in registro_importacion['calendario_presidencial']]
calendario['eleccion_presidencial_col'] = calendario['year'].isin(anios_electorales).astype(int)

## Integrar las variables sin modificar la muestra

In [14]:
base_ml = base_filtrada.merge(datos_destino, on = ['iso3_d', 'year'], how = 'left', validate = 'many_to_one')
base_ml = base_ml.merge(mercado, on = ['hs6', 'iso3_d', 'year'], how = 'left', validate = 'one_to_one')
base_ml['mercado_registrado_baci'] = base_ml['importaciones_producto_total_usd'].notna()
# Sin flujos positivos registrados en BACI, las importaciones y el número de proveedores son cero.
base_ml['importaciones_producto_total_usd'] = base_ml['importaciones_producto_total_usd'].fillna(0)
base_ml['proveedores_ex_col'] = base_ml['proveedores_ex_col'].fillna(0).astype(int)
columnas_contexto = ['hs6', 'year', 'sector_cn', 'estado_concordancia', 'nombre_sector_cn',
    'va_sector_col_miles_millones_cop', 'ci_sector_col_miles_millones_cop', 'va_sector_col_usd', 'ci_sector_col_usd']
base_ml = base_ml.merge(contexto_dane[columnas_contexto], on = ['hs6', 'year'], how = 'left', validate = 'many_to_one')
base_ml = base_ml.merge(oferta[['hs6', 'year', 'participacion_oferta_col']], on = ['hs6', 'year'], how = 'left', validate = 'many_to_one')
base_ml = base_ml.merge(series_anuales, on = 'year', how = 'left', validate = 'many_to_one')
base_ml = base_ml.merge(calendario, on = 'year', how = 'left', validate = 'many_to_one')
assert len(base_ml) == len(base_filtrada) and not base_ml.duplicated(['hs6', 'iso3_d', 'year']).any()
assert base_ml.groupby(['hs6', 'iso3_d'])['year'].nunique().eq(8).all()

In [15]:
base_ml.head()

,iso3_o,hs6,iso3_d,year,trade_value_usd,registro_baci,cero_construido,exportaciones_estimadas_usd,brecha_usd,brecha_relativa,efecto_fijo_finito,estado_ajuste,nombre_producto_hs6_baci,nombre_pais_destino,pib_ppa_constante_2021,poblacion_total,va_industrial_constante_2015_usd,ied_entrada_pct_pib,acceso_electricidad_pct,wgi_estabilidad_politica,importaciones_producto_total_usd,proveedores_ex_col,mercado_registrado_baci,sector_cn,estado_concordancia,nombre_sector_cn,va_sector_col_miles_millones_cop,ci_sector_col_miles_millones_cop,va_sector_col_usd,ci_sector_col_usd,participacion_oferta_col,trm_cop_por_usd,trm_usd_por_cop,precio_petroleo_brent_usd_barril,precio_cafe_arabica_usd_kg,eleccion_presidencial_col
0,COL,010121,ABW,2017,0.00,False,True,421.96,421.96,1.00,True,ajuste_en_muestra,"Horses: live, pure-bred breeding animals",Aruba,"4,155,122,005.38","108,735.00","414,525,108.60",5.25,100.00,1.22,0.00,0,False,A0102,unica,"Ganadería, caza y actividades de servicios con...","14,639.00","16,149.00","4,961,836,144.02","5,473,645,186.81",0.00,"2,951.32",0.00,54.40,3.32,0
1,COL,010121,ABW,2018,0.00,False,True,"1,214.99","1,214.99",1.00,True,ajuste_en_muestra,"Horses: live, pure-bred breeding animals",Aruba,"4,288,604,099.77","108,908.00","416,895,932.37",4.40,100.00,1.34,"6,690.00",1,True,A0102,unica,"Ganadería, caza y actividades de servicios con...","15,224.00","16,788.00","5,161,350,126.73","5,691,588,671.01",0.00,"2,956.43",0.00,71.10,2.93,1
2,COL,010121,ABW,2019,"4,300.00",True,False,435.60,"-3,864.40",-8.87,True,ajuste_en_muestra,"Horses: live, pure-bred breeding animals",Aruba,"4,341,317,223.57","109,203.00","415,300,901.93",-2.25,100.00,1.34,"4,300.00",0,True,A0102,unica,"Ganadería, caza y actividades de servicios con...","15,735.00","17,787.00","4,802,612,775.73","5,428,921,095.76",0.00,"3,281.09",0.00,64.00,2.88,0
3,COL,010121,ABW,2020,0.00,False,True,382.85,382.85,1.00,True,ajuste_en_muestra,"Horses: live, pure-bred breeding animals",Aruba,"3,302,003,307.24","108,587.00","344,415,091.09",6.54,100.00,1.34,0.00,0,False,A0102,unica,"Ganadería, caza y actividades de servicios con...","16,980.00","18,498.00","4,613,056,801.56","5,025,460,819.51",0.00,"3,693.36",0.00,42.30,3.32,0
4,COL,010121,ABW,2021,0.00,False,True,591.10,591.10,1.00,True,ajuste_en_muestra,"Horses: live, pure-bred breeding animals",Aruba,"3,791,812,306.88","107,700.00","361,042,348.73",4.62,100.00,1.34,0.00,0,False,A0102,unica,"Ganadería, caza y actividades de servicios con...","21,931.00","22,560.00","5,868,190,837.58","6,036,495,613.32",0.00,"3,743.09",0.00,70.40,4.51,0


## Seleccionar los quince predictores

In [16]:
predictores = [
    'pib_ppa_constante_2021', 'poblacion_total', 'va_industrial_constante_2015_usd',
    'ied_entrada_pct_pib', 'acceso_electricidad_pct', 'wgi_estabilidad_politica',
    'importaciones_producto_total_usd', 'proveedores_ex_col',
    'precio_petroleo_brent_usd_barril', 'precio_cafe_arabica_usd_kg',
    'participacion_oferta_col', 'va_sector_col_usd', 'ci_sector_col_usd',
    'eleccion_presidencial_col', 'trm_usd_por_cop'
]
# Los valores originales DANE en COP quedan fuera de los predictores.
X_ml = base_ml[predictores].copy()
assert X_ml.shape[1] == 15
pd.Series(predictores, index = range(1, 16), name = 'Variable').to_frame()

,Variable
1,pib_ppa_constante_2021
2,poblacion_total
3,va_industrial_constante_2015_usd
4,ied_entrada_pct_pib
5,acceso_electricidad_pct
6,wgi_estabilidad_politica
7,importaciones_producto_total_usd
8,proveedores_ex_col
9,precio_petroleo_brent_usd_barril
10,precio_cafe_arabica_usd_kg


## Identificar los faltantes de las variables de destino

In [17]:
def patron_temporal(anios, valores):
    orden = np.argsort(np.asarray(anios))
    faltan = pd.isna(np.asarray(valores, dtype=object)[orden])
    if not faltan.any():
        return "completo"
    observados = np.flatnonzero(~faltan)
    if not len(observados):
        return "sin_datos_en_horizonte"
    interior = faltan[observados[0]:observados[-1]+1].any()
    inicial, final = bool(faltan[0]), bool(faltan[-1])
    if interior:
        return "huecos_internos_y_extremos" if inicial or final else "huecos_internos"
    if inicial and final:
        return "ambos_extremos"
    return "faltantes_iniciales" if inicial else "faltantes_finales"

variables_destino = [variable for _, variable in indicadores]
macro_pais_anio = datos_destino.merge(
    catalogo_destinos[['iso3_d', 'nombre_destino']].rename(columns={'nombre_destino':'nombre_pais_destino'}),
    on = 'iso3_d', validate = 'many_to_one'
)
assert not macro_pais_anio.duplicated(['iso3_d', 'year']).any()
historia = []
for (iso, nombre), grupo in macro_pais_anio.groupby(['iso3_d', 'nombre_pais_destino']):
    grupo = grupo.sort_values('year')
    for variable in variables_destino:
        faltan = grupo[variable].isna()
        historia.append({
            'iso3_d': iso, 'nombre_pais_destino': nombre, 'variable': variable,
            'anios': len(grupo), 'faltantes': int(faltan.sum()), 'faltantes_pct': 100 * faltan.mean(),
            'anios_sin_dato': ', '.join(grupo.loc[faltan, 'year'].astype(str)),
            'patron_temporal': patron_temporal(grupo['year'], grupo[variable])
        })
historia = pd.DataFrame(historia)
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(historia.loc[historia.faltantes.gt(0)].sort_values(
        ['variable', 'faltantes', 'iso3_d'], ascending = [True, False, True]))

,iso3_d,nombre_pais_destino,variable,anios,faltantes,faltantes_pct,anios_sin_dato,patron_temporal
52,ASM,Samoa Americana,acceso_electricidad_pct,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
904,S19,Taiwán,acceso_electricidad_pct,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
51,ASM,Samoa Americana,ied_entrada_pct_pib,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
369,FSM,Micronesia (Estados Federados de),ied_entrada_pct_pib,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
435,GRL,Groenlandia,ied_entrada_pct_pib,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
447,GUM,Guam,ied_entrada_pct_pib,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
729,MNP,Islas Marianas Septentrionales,ied_entrada_pct_pib,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
903,S19,Taiwán,ied_entrada_pct_pib,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
1011,SYR,República Árabe Siria,ied_entrada_pct_pib,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
1149,YEM,Yemen,ied_entrada_pct_pib,8,6,75.00,"2019, 2020, 2021, 2022, 2023, 2024",faltantes_finales


## Calcular el cambio promedio con los datos observados

In [18]:
PUNTOS = ['ied_entrada_pct_pib', 'acceso_electricidad_pct', 'wgi_estabilidad_politica']

def tasas_medias(tabla, variables):
    filas=[]
    if tabla.duplicated(['iso3_d','year']).any():raise ValueError('País-año duplicado en el cálculo de crecimiento.')
    for v in variables:
        anterior=tabla[['iso3_d','year',v]].copy()
        anterior['year']+=1
        pares=tabla[['iso3_d','year',v]].merge(anterior,on=['iso3_d','year'],suffixes=('_t','_prev'),validate='one_to_one')
        for y in ANIOS[1:]:
            g=pares.loc[pares.year.eq(y)].dropna(subset=[v+'_t',v+'_prev'])
            if v in PUNTOS:
                valores=g[v+'_t']-g[v+'_prev'];regla='cambio_promedio_puntos'
            else:
                g=g.loc[g[v+'_prev'].gt(0)&g[v+'_t'].ge(0)]
                valores=g[v+'_t']/g[v+'_prev']-1;regla='crecimiento_promedio_relativo'
            valores=valores.loc[np.isfinite(valores)]
            filas.append({'variable':v,'year':y,'donantes':len(valores),'cambio_promedio':float(valores.mean()) if len(valores) else np.nan,'regla':regla})
    return pd.DataFrame(filas)

macro_original = macro_pais_anio.copy()
macro_completada = macro_original.copy()
tasas_promedio = tasas_medias(macro_original, variables_destino)
candidatos = historia.loc[historia['faltantes'].gt(0) & historia['faltantes'].lt(historia['anios'])]

## Completar los faltantes parciales hacia adelante y hacia atrás

In [19]:
tasas_por_anio = tasas_promedio.set_index(['variable', 'year'])
imputaciones_promedio = []
for caso in candidatos.itertuples(index = False):
    grupo = macro_original.loc[macro_original['iso3_d'].eq(caso.iso3_d)].set_index('year')
    serie = grupo[caso.variable].sort_index().copy()
    primer_anio = int(serie.first_valid_index())
    recorridos = [(1, range(primer_anio + 1, max(ANIOS) + 1)),
                  (-1, range(primer_anio - 1, min(ANIOS) - 1, -1))]
    for paso, anios in recorridos:
        for anio in anios:
            if pd.notna(serie.loc[anio]):
                continue
            anio_ancla = anio - paso
            valor_ancla = float(serie.loc[anio_ancla])
            anio_cambio = max(anio, anio_ancla)
            tasa = tasas_por_anio.loc[(caso.variable, anio_cambio)]
            cambio = float(tasa['cambio_promedio'])
            assert np.isfinite(valor_ancla) and np.isfinite(cambio) and tasa['donantes'] > 0
            if caso.variable in PUNTOS:
                valor = valor_ancla + paso * cambio
            else:
                assert 1 + cambio > 0
                valor = valor_ancla * (1 + cambio) ** paso
            if caso.variable == 'acceso_electricidad_pct':
                valor = float(np.clip(valor, 0, 100))
            assert np.isfinite(valor)
            serie.loc[anio] = valor
            imputaciones_promedio.append({
                'iso3_d': caso.iso3_d, 'nombre_pais_destino': caso.nombre_pais_destino,
                'variable': caso.variable, 'year': anio,
                'direccion': 'hacia_adelante' if paso == 1 else 'hacia_atras',
                'anio_ancla': anio_ancla, 'valor_ancla': valor_ancla,
                'anio_cambio': anio_cambio, 'cambio_promedio': cambio,
                'donantes': int(tasa['donantes']), 'regla': tasa['regla'], 'valor_imputado': valor
            })
    mascara = macro_completada['iso3_d'].eq(caso.iso3_d)
    macro_completada.loc[mascara, caso.variable] = macro_completada.loc[mascara, 'year'].map(serie)
    mascara = base_ml['iso3_d'].eq(caso.iso3_d)
    base_ml.loc[mascara, caso.variable] = base_ml.loc[mascara, 'year'].map(serie)
imputaciones_promedio = pd.DataFrame(imputaciones_promedio)
imputaciones_promedio['crecimiento_promedio_pct'] = np.where(
    imputaciones_promedio['regla'].eq('crecimiento_promedio_relativo'),
    100 * imputaciones_promedio['cambio_promedio'], np.nan)
imputaciones_promedio['cambio_medio_puntos'] = np.where(
    imputaciones_promedio['regla'].eq('cambio_promedio_puntos'),
    imputaciones_promedio['cambio_promedio'], np.nan)

Se excluyen de forma arbitraria los países con alguna variable de destino sin datos durante todo 2017-2024. Se conservan como excepciones Siria (SYR), Cuba (CUB), Venezuela (VEN), Yemen (YEM), Curazao (CUW) y Estados Unidos (USA), cuyos casos se revisarán por separado. Esta decisión incluye la eliminación de Taiwán (S19). La imputación KNN de VA y CI se realiza por separado en el auxiliar 08.1.

## Excluir los países con series completamente ausentes

In [20]:
excepciones = ['SYR', 'CUB', 'VEN', 'YEM', 'CUW', 'USA']
series_sin_datos = macro_completada[variables_destino].isna().groupby([
    macro_completada['iso3_d'], macro_completada['nombre_pais_destino']
]).all().reset_index().melt(id_vars=['iso3_d', 'nombre_pais_destino'],
    value_vars=variables_destino, var_name='variable', value_name='sin_datos')
series_sin_datos = series_sin_datos.loc[series_sin_datos['sin_datos'] & ~series_sin_datos['iso3_d'].isin(excepciones)]
paises_eliminados = series_sin_datos.groupby(['iso3_d', 'nombre_pais_destino'], as_index = False).agg(
    variables_sin_datos = ('variable', lambda x: ', '.join(sorted(x))))
codigos_eliminados = paises_eliminados['iso3_d'].tolist()
filas_antes_exclusion = len(base_ml)
base_ml = base_ml.loc[~base_ml['iso3_d'].isin(codigos_eliminados)].copy().reset_index(drop = True)
datos_destino = macro_completada.loc[~macro_completada['iso3_d'].isin(codigos_eliminados)].drop(columns='nombre_pais_destino').reset_index(drop=True)
X_ml = base_ml[predictores].copy()
assert 'S19' not in set(base_ml['iso3_d'])
assert set(excepciones).issubset(set(base_ml['iso3_d']))
assert base_ml.groupby(['hs6', 'iso3_d'])['year'].nunique().eq(8).all()
display(paises_eliminados)

,iso3_d,nombre_pais_destino,variables_sin_datos
0,ASM,Samoa Americana,"acceso_electricidad_pct, ied_entrada_pct_pib, ..."
1,FSM,Micronesia (Estados Federados de),ied_entrada_pct_pib
2,GRL,Groenlandia,ied_entrada_pct_pib
3,GUM,Guam,"ied_entrada_pct_pib, pib_ppa_constante_2021, v..."
4,MNP,Islas Marianas Septentrionales,"ied_entrada_pct_pib, pib_ppa_constante_2021, v..."
5,PYF,Polinesia Francesa,"pib_ppa_constante_2021, va_industrial_constant..."
6,S19,Taiwán,"acceso_electricidad_pct, ied_entrada_pct_pib, ..."
7,SOM,Somalia,va_industrial_constante_2015_usd
8,TCA,Islas Turcas y Caicos,wgi_estabilidad_politica


In [21]:
macro_actual = base_ml[
    ['iso3_d', 'nombre_pais_destino', 'year', *variables_destino]
].drop_duplicates()

assert not macro_actual.duplicated(['iso3_d', 'year']).any()

historia_actual = []

for (iso, nombre), grupo in macro_actual.groupby(
    ['iso3_d', 'nombre_pais_destino']
):
    grupo = grupo.sort_values('year')

    for variable in variables_destino:
        faltan = grupo[variable].isna()

        historia_actual.append({
            'iso3_d': iso,
            'nombre_pais_destino': nombre,
            'variable': variable,
            'anios': len(grupo),
            'faltantes': int(faltan.sum()),
            'faltantes_pct': 100 * faltan.mean(),
            'anios_sin_dato': ', '.join(
                grupo.loc[faltan, 'year'].astype(str)
            ),
            'patron_temporal': patron_temporal(
                grupo['year'], grupo[variable]
            )
        })

historia_actual = pd.DataFrame(historia_actual)

with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(
        historia_actual.loc[historia_actual.faltantes.gt(0)]
        .sort_values(
            ['variable', 'faltantes', 'iso3_d'],
            ascending=[True, False, True]
        )
    )

,iso3_d,nombre_pais_destino,variable,anios,faltantes,faltantes_pct,anios_sin_dato,patron_temporal
963,SYR,República Árabe Siria,ied_entrada_pct_pib,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
246,CUB,Cuba,pib_ppa_constante_2021,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
1068,VEN,Venezuela (República Bolivariana de),pib_ppa_constante_2021,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
1092,YEM,Yemen,pib_ppa_constante_2021,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
254,CUW,Curazao,va_industrial_constante_2015_usd,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
1052,USA,Estados Unidos de América,va_industrial_constante_2015_usd,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
1070,VEN,Venezuela (República Bolivariana de),va_industrial_constante_2015_usd,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte
257,CUW,Curazao,wgi_estabilidad_politica,8,8,100.00,"2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024",sin_datos_en_horizonte


## Completar el VA industrial de Estados Unidos

In [22]:
industria_usa = pd.read_csv(RAW / 'variables_ml/complementos/usa_va_industrial_2017_2024.csv',
    float_precision = 'round_trip')
assert industria_usa['iso3_d'].eq('USA').all() and set(industria_usa['year']) == set(ANIOS)
assert not industria_usa.duplicated(['iso3_d', 'year']).any()
serie_usa = industria_usa.set_index(['iso3_d', 'year'])['va_industrial_constante_2015_usd']
# El complemento no entra en macro_original: los donantes regionales siguen siendo datos originales.
for tabla in [base_ml, macro_completada, macro_actual, datos_destino]:
    valores = serie_usa.reindex(pd.MultiIndex.from_frame(tabla[['iso3_d', 'year']])).to_numpy()
    tabla['va_industrial_constante_2015_usd'] = tabla['va_industrial_constante_2015_usd'].fillna(
        pd.Series(valores, index = tabla.index))
industria_usa

,iso3_d,year,va_industrial_constante_2015_usd
0,USA,2017,"3,467,730,441,672.64"
1,USA,2018,"3,597,342,058,113.87"
2,USA,2019,"3,665,576,154,691.27"
3,USA,2020,"3,577,309,704,827.69"
4,USA,2021,"3,741,760,139,083.35"
5,USA,2022,"3,638,400,063,352.46"
6,USA,2023,"3,793,098,953,688.47"
7,USA,2024,"3,940,512,331,475.78"


## Calcular los promedios regionales

In [23]:
regiones = catalogo_destinos[['iso3_d', 'region_m49']]
base_regional = macro_original.loc[macro_original['iso3_d'].isin(base_ml['iso3_d'])].merge(
    regiones, on = 'iso3_d', validate = 'many_to_one')
base_regional = base_regional.melt(id_vars = ['iso3_d', 'region_m49', 'year'],
    value_vars = variables_destino, var_name = 'variable', value_name = 'valor')
promedios_regionales = base_regional.groupby(['region_m49', 'variable', 'year'], as_index = False).agg(
    valor_imputado = ('valor', 'mean'), donantes = ('valor', 'count'))

imputaciones_region = macro_actual.melt(id_vars = ['iso3_d', 'nombre_pais_destino', 'year'],
    value_vars = variables_destino, var_name = 'variable', value_name = 'valor')
imputaciones_region = imputaciones_region.loc[imputaciones_region['valor'].isna()].drop(columns = 'valor')
imputaciones_region = imputaciones_region.merge(regiones, on = 'iso3_d', validate = 'many_to_one').merge(
    promedios_regionales, on = ['region_m49', 'variable', 'year'], how = 'left', validate = 'many_to_one')
assert imputaciones_region['valor_imputado'].notna().all() and imputaciones_region['donantes'].gt(0).all()
display(imputaciones_region.sort_values(['variable', 'iso3_d', 'year']).reset_index(drop = True))

,iso3_d,nombre_pais_destino,year,variable,region_m49,valor_imputado,donantes
0,SYR,República Árabe Siria,2017,ied_entrada_pct_pib,Asia,8.26,47
1,SYR,República Árabe Siria,2018,ied_entrada_pct_pib,Asia,10.34,47
2,SYR,República Árabe Siria,2019,ied_entrada_pct_pib,Asia,13.08,46
3,SYR,República Árabe Siria,2020,ied_entrada_pct_pib,Asia,-3.95,46
4,SYR,República Árabe Siria,2021,ied_entrada_pct_pib,Asia,4.84,46
5,SYR,República Árabe Siria,2022,ied_entrada_pct_pib,Asia,4.12,45
6,SYR,República Árabe Siria,2023,ied_entrada_pct_pib,Asia,2.81,45
7,SYR,República Árabe Siria,2024,ied_entrada_pct_pib,Asia,1.06,45
8,CUB,Cuba,2017,pib_ppa_constante_2021,Américas,"938,865,302,868.27",36
9,CUB,Cuba,2018,pib_ppa_constante_2021,Américas,"962,924,860,291.32",36


## Imputar los valores regionales

In [24]:
valores_region = imputaciones_region.pivot(index = ['iso3_d', 'year'],
    columns = 'variable', values = 'valor_imputado')
for tabla in [base_ml, macro_completada, macro_actual, datos_destino]:
    claves = pd.MultiIndex.from_frame(tabla[['iso3_d', 'year']])
    for variable in valores_region.columns:
        valores = valores_region[variable].reindex(claves).to_numpy()
        tabla[variable] = tabla[variable].fillna(pd.Series(valores, index = tabla.index))
X_ml = base_ml[predictores].copy()

faltantes_destino = macro_actual[variables_destino].isna().sum().rename('faltantes').to_frame()
display(faltantes_destino)
assert faltantes_destino['faltantes'].eq(0).all()

,faltantes
pib_ppa_constante_2021,0
poblacion_total,0
va_industrial_constante_2015_usd,0
ied_entrada_pct_pib,0
acceso_electricidad_pct,0
wgi_estabilidad_politica,0


## Asignar cero a la brecha relativa 0/0

In [25]:
cero_cero = (base_ml['brecha_relativa'].isna() & base_ml['trade_value_usd'].eq(0)
    & base_ml['exportaciones_estimadas_usd'].eq(0))
base_ml.loc[cero_cero, 'brecha_relativa'] = 0
pd.Series({'Brechas 0/0 asignadas a cero': int(cero_cero.sum()),
    'Brechas relativas aún faltantes': int(base_ml['brecha_relativa'].isna().sum())})

Brechas 0/0 asignadas a cero       17013
Brechas relativas aún faltantes        0
dtype: int64

## Guardar la base preparada

In [26]:
archivo_base_ml = PROCESSED / 'ml/base_ml_06.parquet'
archivo_base_ml.parent.mkdir(parents = True, exist_ok = True)
base_ml.to_parquet(archivo_base_ml, index = False)